<a href="https://colab.research.google.com/github/GAIA-HKUSTGZ/UCUG1002_AI-Society/blob/main/labs/2_Network/lab_a_text_to_cypher_graph_qa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab A: Text-to-Cypher Graph Database QA

**Course:** AI + Society  
**Topic:** Natural language questions over a social knowledge graph  


In this lab, you will build the most teachable version of graph QA:

1. A user asks a natural-language question.
2. The LLM reads the graph schema.
3. The LLM writes a Cypher query.
4. Neo4j executes the graph query.
5. The result is summarized as a natural-language answer.

The notebook is **LLM-first with an offline fallback**:

- If an LLM API key is available, the notebook asks the LLM to generate Cypher and, in the real Neo4j path, to summarize the result.
- If no API key is available, or if `USE_LLM_REASONING = False`, it falls back to the transparent offline demo path.
- If Neo4j Aura credentials are available, the generated Cypher can run against Neo4j; otherwise it runs against the classroom mock executor.

## How to open this lab in Google Colab

Recommended classroom path:

1. Open the Colab link shared by your instructor.
2. Click **File -> Save a copy in Drive** to create your own editable copy.
3. Add your API key in Colab **Secrets** if you want to use a real LLM.
4. Run cells from top to bottom, or use **Runtime -> Run all**.

Fallback path if your instructor gives you only the `.ipynb` file:

1. Go to Google Colab: https://colab.research.google.com/
2. Choose **Upload**.
3. Click **Choose file**, then select this notebook file.
4. After it opens, click **File -> Save a copy in Drive**.

If you want to use a real LLM, add your API key in Colab **Secrets** before running the LLM cells.

## Student roadmap

This lab has one central idea: an LLM can translate a natural-language question into a graph query, but the query should remain visible and inspectable.

| Part | What you run | What you should understand | What to observe |
|---|---|---|---|
| Setup | Install/import packages and configure LLM provider | A notebook needs libraries and credentials before it can talk to an LLM or database | The lab can still run without keys because it has a fallback path |
| Schema | Read the graph schema | The LLM needs labels, relationships, and property names before writing Cypher | Wrong schema usually means wrong query |
| Toy graph | Build people, policies, regions, events, and tools | A knowledge graph stores entities and relationships, not just text chunks | The same social question can become a path query |
| Visualization | Draw the full graph and a query subgraph | Visual inspection helps connect Cypher syntax to graph structure | The query subgraph is much smaller than the full graph |
| Text-to-Cypher | Ask questions and inspect generated Cypher | The LLM writes a structured query; the database returns rows | Do not trust only the final answer; inspect the Cypher |
| Neo4j path | Optional real database execution | The same logic can run against a real graph database | Credentials are URI + username + password, not a typical API key |
| Guardrails | Check read-only queries | LLM-generated queries need safety checks | Some Cypher commands can modify or delete data |

In [ ]:
# Colab setup. This cell is safe to run, but the real Neo4j path remains optional.
import importlib
import subprocess
import sys

required = {
    "pandas": "pandas",
    "networkx": "networkx",
    "matplotlib": "matplotlib",
    "openai": "openai",
    "requests": "requests",
    "neo4j": "neo4j",
    "langchain": "langchain",
    "langchain_neo4j": "langchain-neo4j",
    "langchain_openai": "langchain-openai",
}

for module, package in required.items():
    try:
        importlib.import_module(module)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import os
import re
import json
import getpass
import textwrap
import requests
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 90)
plt.rcParams["figure.figsize"] = (11, 7)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
print("Setup complete.")

## 0. API key setup

This notebook is LLM-first, but it still runs without a key. Students should **not paste API keys directly into code cells**. Use one of these two safer options:

### Get a Gemini API key

1. Visit Google AI Studio: https://aistudio.google.com/
2. Sign in with your Google account.
3. Click **Get API key** in the left sidebar, then click **Create API key**.
4. Select a Google Cloud project. If you do not have one, Google AI Studio can create a default project for you.
5. Copy the API key.

### Add the key to Colab

1. In Colab, open **Secrets** in the left sidebar.
2. Add a new secret named `GEMINI_API_KEY`.
3. Paste your copied key as the value.
4. Keep notebook access enabled for this secret.

OpenAI is also supported: add `OPENAI_API_KEY` instead of `GEMINI_API_KEY`.

If you are not using Colab Secrets, set `PROMPT_FOR_API_KEY = True` and paste the key into the hidden password prompt.

The key is stored only in the current notebook runtime environment as `os.environ["GEMINI_API_KEY"]` or `os.environ["OPENAI_API_KEY"]`.

In [ ]:
LLM_PROVIDER = os.environ.get("LLM_PROVIDER", "auto").lower()  # "auto", "gemini", or "openai"
GEMINI_MODEL = os.environ.get("GEMINI_MODEL", "gemini-2.5-flash")
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
USE_LLM_REASONING = True
PROMPT_FOR_API_KEY = False  # Keep False for automatic fallback; set True to ask for a hidden key prompt.


def _load_colab_secret(secret_name):
    try:
        from google.colab import userdata

        return userdata.get(secret_name)
    except Exception:
        return None


def try_load_provider_key(provider, prompt_if_missing=False):
    provider = provider.lower()
    env_name = "GEMINI_API_KEY" if provider == "gemini" else "OPENAI_API_KEY"
    if os.environ.get(env_name):
        return True

    key = _load_colab_secret(env_name)
    if key:
        os.environ[env_name] = key
        return True

    if not prompt_if_missing:
        return False

    label = "Gemini" if provider == "gemini" else "OpenAI"
    key = getpass.getpass(f"Paste your {label} API key. It will not be printed: ").strip()
    if not key:
        return False
    os.environ[env_name] = key
    return True


def resolve_llm_provider(prompt_if_missing=False):
    if LLM_PROVIDER in {"gemini", "google"}:
        return "gemini" if try_load_provider_key("gemini", prompt_if_missing) else None
    if LLM_PROVIDER == "openai":
        return "openai" if try_load_provider_key("openai", prompt_if_missing) else None

    # Auto mode: prefer Gemini when a Gemini key exists, otherwise OpenAI.
    if try_load_provider_key("gemini", False):
        return "gemini"
    if try_load_provider_key("openai", False):
        return "openai"
    if prompt_if_missing:
        return "gemini" if try_load_provider_key("gemini", True) else None
    return None


def ensure_llm_api_key():
    provider = resolve_llm_provider(prompt_if_missing=True)
    if not provider:
        raise ValueError("No Gemini or OpenAI API key was provided.")
    print(f"{provider} API key is available for this runtime.")
    return provider


def llm_text(prompt, temperature=0):
    provider = ensure_llm_api_key()
    if provider == "gemini":
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent"
        headers = {
            "x-goog-api-key": os.environ["GEMINI_API_KEY"],
            "Content-Type": "application/json",
        }
        body = {
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {"temperature": temperature, "maxOutputTokens": 1024},
        }
        response = requests.post(url, headers=headers, json=body, timeout=60)
        response.raise_for_status()
        data = response.json()
        parts = data["candidates"][0]["content"].get("parts", [])
        return "".join(part.get("text", "") for part in parts).strip()

    from openai import OpenAI

    client = OpenAI()
    response = client.responses.create(model=OPENAI_MODEL, input=prompt, temperature=temperature)
    return response.output_text.strip()


print(f"LLM_PROVIDER = {LLM_PROVIDER}")
print(f"Gemini model: {GEMINI_MODEL}; OpenAI model: {OPENAI_MODEL}")
print(f"USE_LLM_REASONING = {USE_LLM_REASONING}")
print("If no API key is found, the notebook falls back to the offline path.")

## 1. The mental model

Text-to-Cypher is not normal top-k document retrieval. It is a **structured query workflow**.

```text
Question
  -> schema-aware Cypher generation
  -> database execution
  -> returned nodes / paths / relations
  -> answer synthesis
```

The schema matters because it tells the model which node labels and relationship types actually exist. Without a schema, the model may invent labels, relationships, or properties.

In [ ]:
SCHEMA_DESCRIPTION = '''
Node labels:
  Person(name, role, activity)
  Organization(name, sector)
  Policy(name, category, year)
  Region(name, risk_level, flood_risk)
  Event(name, event_type, date)
  Tool(name, method)

Relationship types:
  (Policy)-[:AFFECTS {strength}]->(Region)
  (Organization)-[:IMPLEMENTS]->(Policy)
  (Person)-[:MEMBER_OF]->(Organization)
  (Person)-[:INFLUENCES {weight}]->(Person)
  (Organization)-[:COORDINATES_WITH]->(Organization)
  (Event)-[:OCCURRED_IN]->(Region)
  (Event)-[:PRECEDES]->(Event)
  (Event)-[:TRIGGERS]->(Policy)
  (Policy)-[:USES_AI_TOOL]->(Tool)
  (Tool)-[:SUPPORTS]->(Organization)
  (Tool)-[:DETECTS]->(Event)
'''

print(SCHEMA_DESCRIPTION)

## 2. A tiny AI + Society graph

The graph is intentionally small. It is about disaster response, misinformation, public agencies, policies, regions, and AI tools. Small graphs are useful for teaching because students can inspect every relationship.

In [ ]:
COURSE_TAG = "AI_SOCIETY_LAB_TEXT_TO_CYPHER"

REGIONS = [
    {"name": "Harbor District", "risk_level": "high", "flood_risk": 0.85, "course": COURSE_TAG},
    {"name": "River County", "risk_level": "medium-high", "flood_risk": 0.65, "course": COURSE_TAG},
    {"name": "Inland Campus", "risk_level": "low", "flood_risk": 0.20, "course": COURSE_TAG},
    {"name": "Old Town", "risk_level": "medium", "flood_risk": 0.45, "course": COURSE_TAG},
]

POLICIES = [
    {"name": "Evacuation Order", "category": "emergency management", "year": 2025, "course": COURSE_TAG},
    {"name": "Road Closure Plan", "category": "transportation", "year": 2025, "course": COURSE_TAG},
    {"name": "Misinformation Response", "category": "information governance", "year": 2025, "course": COURSE_TAG},
    {"name": "AI Damage Assessment", "category": "AI-assisted response", "year": 2025, "course": COURSE_TAG},
]

ORGANIZATIONS = [
    {"name": "Emergency Agency", "sector": "government", "course": COURSE_TAG},
    {"name": "Transit Authority", "sector": "public transport", "course": COURSE_TAG},
    {"name": "University Lab", "sector": "research", "course": COURSE_TAG},
    {"name": "Community Volunteers", "sector": "civil society", "course": COURSE_TAG},
]

PEOPLE = [
    {"name": "Mayor Chen", "role": "public official", "activity": "high", "course": COURSE_TAG},
    {"name": "Prof. Rivera", "role": "scientist", "activity": "medium", "course": COURSE_TAG},
    {"name": "Local Influencer", "role": "media actor", "activity": "high", "course": COURSE_TAG},
    {"name": "Resident Group", "role": "community actor", "activity": "medium", "course": COURSE_TAG},
]

EVENTS = [
    {"name": "Hurricane Iris", "event_type": "hazard", "date": "2025-08-18", "course": COURSE_TAG},
    {"name": "Rumor Surge", "event_type": "information event", "date": "2025-08-18", "course": COURSE_TAG},
    {"name": "Bridge Flooding", "event_type": "infrastructure disruption", "date": "2025-08-19", "course": COURSE_TAG},
]

TOOLS = [
    {"name": "Damage Classifier", "method": "computer vision", "course": COURSE_TAG},
    {"name": "Graph Alert Agent", "method": "graph reasoning", "course": COURSE_TAG},
]

RELATIONSHIPS = {
    "AFFECTS": [
        ("Evacuation Order", "Harbor District", {"strength": 0.90}),
        ("Evacuation Order", "River County", {"strength": 0.75}),
        ("Road Closure Plan", "Harbor District", {"strength": 0.80}),
        ("Misinformation Response", "Inland Campus", {"strength": 0.55}),
    ],
    "IMPLEMENTS": [
        ("Emergency Agency", "Evacuation Order", {}),
        ("Transit Authority", "Road Closure Plan", {}),
        ("Emergency Agency", "Misinformation Response", {}),
        ("University Lab", "AI Damage Assessment", {}),
    ],
    "MEMBER_OF": [
        ("Mayor Chen", "Emergency Agency", {}),
        ("Prof. Rivera", "University Lab", {}),
        ("Resident Group", "Community Volunteers", {}),
    ],
    "INFLUENCES": [
        ("Local Influencer", "Resident Group", {"weight": 0.70}),
        ("Mayor Chen", "Local Influencer", {"weight": 0.45}),
    ],
    "COORDINATES_WITH": [
        ("Emergency Agency", "Transit Authority", {}),
        ("Emergency Agency", "University Lab", {}),
        ("Community Volunteers", "Emergency Agency", {}),
    ],
    "OCCURRED_IN": [
        ("Hurricane Iris", "River County", {}),
        ("Rumor Surge", "Inland Campus", {}),
        ("Bridge Flooding", "Harbor District", {}),
    ],
    "PRECEDES": [
        ("Hurricane Iris", "Bridge Flooding", {}),
        ("Rumor Surge", "Misinformation Response", {}),
    ],
    "TRIGGERS": [
        ("Hurricane Iris", "Evacuation Order", {}),
        ("Bridge Flooding", "Road Closure Plan", {}),
        ("Rumor Surge", "Misinformation Response", {}),
    ],
    "USES_AI_TOOL": [
        ("AI Damage Assessment", "Damage Classifier", {}),
        ("Misinformation Response", "Graph Alert Agent", {}),
    ],
    "SUPPORTS": [
        ("Damage Classifier", "Emergency Agency", {}),
        ("Graph Alert Agent", "Emergency Agency", {}),
    ],
    "DETECTS": [
        ("Damage Classifier", "Bridge Flooding", {}),
    ],
}

node_preview = pd.DataFrame(REGIONS + POLICIES + ORGANIZATIONS + PEOPLE + EVENTS + TOOLS)
node_preview[["name", "course"]].head(12)

In [ ]:
relationship_preview = []
for rel_type, rows in RELATIONSHIPS.items():
    for source, target, props in rows:
        relationship_preview.append({"source": source, "relationship": rel_type, "target": target, **props})

pd.DataFrame(relationship_preview)

## 3. Visualize the classroom graph

Before asking the LLM to write Cypher, inspect the graph visually. The goal is not to make a perfect network diagram; it is to see what kinds of nodes and relationships the query can traverse.

In [ ]:
def build_visual_graph():
    graph = nx.DiGraph()

    typed_nodes = [
        ("Region", REGIONS),
        ("Policy", POLICIES),
        ("Organization", ORGANIZATIONS),
        ("Person", PEOPLE),
        ("Event", EVENTS),
        ("Tool", TOOLS),
    ]
    for node_type, rows in typed_nodes:
        for row in rows:
            graph.add_node(row["name"], node_type=node_type)

    for rel_type, rows in RELATIONSHIPS.items():
        for source, target, props in rows:
            graph.add_edge(source, target, relationship=rel_type, **props)

    return graph


NODE_COLORS = {
    "Region": "#b7d7f0",
    "Policy": "#f6c85f",
    "Organization": "#9bd18b",
    "Person": "#c8b6e2",
    "Event": "#f4a6a6",
    "Tool": "#b9b9b9",
}


def draw_graph(graph, highlight_nodes=None, highlight_edges=None, title="AI + Society toy knowledge graph"):
    highlight_nodes = set(highlight_nodes or [])
    highlight_edges = set(highlight_edges or [])
    pos = nx.spring_layout(graph, seed=8, k=1.15)

    node_colors = [
        "#ffdf80" if node in highlight_nodes else NODE_COLORS.get(graph.nodes[node].get("node_type"), "#dddddd")
        for node in graph.nodes
    ]
    edge_colors = [
        "#d62728" if (source, target) in highlight_edges else "#8492a6"
        for source, target in graph.edges
    ]
    edge_widths = [
        2.8 if (source, target) in highlight_edges else 1.2
        for source, target in graph.edges
    ]

    plt.figure(figsize=(12, 7))
    nx.draw_networkx_nodes(graph, pos, node_color=node_colors, edgecolors="#263238", node_size=1400)
    nx.draw_networkx_labels(graph, pos, font_size=8)
    nx.draw_networkx_edges(graph, pos, edge_color=edge_colors, width=edge_widths, arrows=True, arrowsize=16)
    edge_labels = {(u, v): data["relationship"] for u, v, data in graph.edges(data=True)}
    nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels, font_size=7)
    plt.title(title)
    plt.axis("off")
    plt.show()


KG_VIS = build_visual_graph()
draw_graph(KG_VIS)

## 4. Visualize a query subgraph

For the question **"Which policies affect Harbor District?"**, the relevant structure is small: policy nodes connected by `AFFECTS` edges to the `Harbor District` region.

In [ ]:
harbor_edges = [
    (source, target)
    for source, target, data in KG_VIS.edges(data=True)
    if data["relationship"] == "AFFECTS" and target == "Harbor District"
]
harbor_nodes = {"Harbor District"} | {source for source, target in harbor_edges}
harbor_subgraph = KG_VIS.subgraph(harbor_nodes).copy()

draw_graph(
    harbor_subgraph,
    highlight_nodes=harbor_nodes,
    highlight_edges=harbor_edges,
    title="Query subgraph: policies affecting Harbor District",
)

## 5. LLM-first demo: question -> Cypher -> result -> answer

This section tries to use a real LLM first. If neither `GEMINI_API_KEY` nor `OPENAI_API_KEY` is available, or if `USE_LLM_REASONING = False`, it automatically falls back to deterministic offline mappings.

This lets the classroom workflow stay reliable while still making the default path a real LLM-based Text-to-Cypher demonstration.

In [ ]:
DANGEROUS_CYPHER = re.compile(
    r"\b(CREATE|MERGE|DELETE|DETACH|SET|REMOVE|DROP|ALTER|LOAD\s+CSV|CALL\s+dbms)\b",
    re.IGNORECASE,
)


def looks_read_only(cypher):
    text = cypher.strip()
    if DANGEROUS_CYPHER.search(text):
        return False
    return text.upper().startswith(("MATCH", "WITH", "RETURN", "CALL DB."))


def clean_cypher(text):
    text = text.strip()
    fence = re.search(r"```(?:cypher)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if fence:
        text = fence.group(1).strip()
    if text.lower().startswith("cypher"):
        text = text[6:].strip()
    return text.strip()


def offline_generate_cypher(question):
    q = question.lower()
    if ("policy" in q or "policies" in q) and "harbor" in q:
        return '''
MATCH (p:Policy)-[r:AFFECTS]->(reg:Region {name: "Harbor District"})
RETURN p.name AS policy, p.category AS category, r.strength AS strength
ORDER BY strength DESC
'''
    if "organization" in q and "river county" in q:
        return '''
MATCH (org:Organization)-[:IMPLEMENTS]->(p:Policy)-[:AFFECTS]->(reg:Region {name: "River County"})
RETURN org.name AS organization, p.name AS policy, reg.name AS region
'''
    if "event" in q and "before" in q and "bridge flooding" in q:
        return '''
MATCH (e1:Event)-[:PRECEDES]->(e2:Event {name: "Bridge Flooding"})
RETURN e1.name AS earlier_event, e1.date AS date, e2.name AS later_event
ORDER BY date
'''
    if "ai" in q or "tool" in q:
        return '''
MATCH (p:Policy)-[:USES_AI_TOOL]->(t:Tool)-[:SUPPORTS]->(org:Organization)
RETURN p.name AS policy, t.name AS ai_tool, t.method AS method, org.name AS supported_organization
'''
    return '''
MATCH (n)
RETURN n.name AS name
LIMIT 5
'''


def offline_execute_cypher(cypher):
    normalized = " ".join(cypher.split()).lower()
    if "harbor district" in normalized and "affects" in normalized:
        return [
            {"policy": "Evacuation Order", "category": "emergency management", "strength": 0.90},
            {"policy": "Road Closure Plan", "category": "transportation", "strength": 0.80},
        ]
    if "river county" in normalized and "implements" in normalized:
        return [
            {"organization": "Emergency Agency", "policy": "Evacuation Order", "region": "River County"},
        ]
    if "precedes" in normalized and "bridge flooding" in normalized:
        return [
            {"earlier_event": "Hurricane Iris", "date": "2025-08-18", "later_event": "Bridge Flooding"},
        ]
    if "uses_ai_tool" in normalized:
        return [
            {
                "policy": "AI Damage Assessment",
                "ai_tool": "Damage Classifier",
                "method": "computer vision",
                "supported_organization": "Emergency Agency",
            },
            {
                "policy": "Misinformation Response",
                "ai_tool": "Graph Alert Agent",
                "method": "graph reasoning",
                "supported_organization": "Emergency Agency",
            },
        ]
    return [{"name": "Harbor District"}, {"name": "Emergency Agency"}, {"name": "Evacuation Order"}]


def offline_summarize(question, rows):
    if not rows:
        return "No matching graph evidence was found."
    cols = rows[0].keys()
    compact = "; ".join(", ".join(f"{k}: {row[k]}" for k in cols) for row in rows)
    return f"Based on the graph query, the answer is: {compact}."


def llm_generate_cypher(question):
    prompt = f'''
You are a careful Text-to-Cypher assistant for an undergraduate AI + Society lab.

Use only this schema:
{SCHEMA_DESCRIPTION}

Known entity names include:
Regions: Harbor District, River County, Inland Campus, Old Town
Policies: Evacuation Order, Road Closure Plan, Misinformation Response, AI Damage Assessment
Organizations: Emergency Agency, Transit Authority, University Lab, Community Volunteers
Events: Hurricane Iris, Rumor Surge, Bridge Flooding
Tools: Damage Classifier, Graph Alert Agent

Task:
Convert the question into one read-only Cypher query.

Rules:
- Return only the Cypher query.
- Use MATCH / RETURN only.
- Do not use CREATE, MERGE, DELETE, SET, REMOVE, DROP, ALTER, or LOAD CSV.
- Use property names from the schema.
- Prefer concise query results with readable aliases.

Question: {question}
'''
    return clean_cypher(llm_text(prompt, temperature=0))


def llm_summarize_graph_result(question, cypher, rows):
    prompt = f'''
You are answering a student's graph database question.

Question:
{question}

Cypher query:
{cypher}

Returned rows:
{json.dumps(rows, indent=2)}

Write a concise natural-language answer. Mention that the answer is based on the returned graph rows.
'''
    return llm_text(prompt, temperature=0)


def offline_graph_qa(question):
    cypher = offline_generate_cypher(question)
    rows = offline_execute_cypher(cypher)
    answer = offline_summarize(question, rows)
    return {"mode": "offline_fallback", "question": question, "cypher": cypher.strip(), "rows": rows, "answer": answer}


def graph_qa_llm_first(question, use_llm=USE_LLM_REASONING, prompt_for_key=PROMPT_FOR_API_KEY):
    use_real_llm = use_llm and resolve_llm_provider(prompt_if_missing=prompt_for_key)
    if use_real_llm:
        try:
            cypher = llm_generate_cypher(question)
            if not looks_read_only(cypher):
                raise ValueError(f"Generated Cypher did not pass the read-only guardrail: {cypher}")
            rows = offline_execute_cypher(cypher)
            answer = llm_summarize_graph_result(question, cypher, rows)
            return {"mode": "llm_cypher_with_mock_execution", "question": question, "cypher": cypher, "rows": rows, "answer": answer}
        except Exception as exc:
            print(f"LLM path failed or was unavailable ({exc}). Falling back to offline demo.")

    return offline_graph_qa(question)

In [ ]:
demo_questions = [
    "Which policies affect Harbor District?",
    "Which organization implements a policy affecting River County?",
    "What events happened before Bridge Flooding?",
    "Which AI tools support emergency response policies?",
]

for question in demo_questions:
    result = graph_qa_llm_first(question)
    print("=" * 90)
    print("MODE:", result["mode"])
    print("QUESTION:", result["question"])
    print("\nGENERATED CYPHER:\n" + result["cypher"])
    print("\nDATABASE RESULT:")
    display(pd.DataFrame(result["rows"]))
    print("ANSWER:", result["answer"])

## 6. Optional real Neo4j path: create AuraDB credentials

Neo4j Aura does not normally give you an "API key" for this Python driver workflow. Instead, you connect with three values:

- `NEO4J_URI`: the connection URI, usually like `neo4j+s://xxxxx.databases.neo4j.io`
- `NEO4J_USERNAME`: often `neo4j`, unless you created another database user
- `NEO4J_PASSWORD`: the database password created with the AuraDB instance

### Create a Neo4j AuraDB instance

1. Go to Neo4j Aura Console: https://console.neo4j.io/
2. Sign in or create a Neo4j account.
3. Create an **AuraDB** instance. For a class lab, **AuraDB Free** is enough if it is available for your account.
4. Choose an empty instance or blank database.
5. When the instance is created, save or download the credentials file. The password is shown during creation and may not be recoverable later.
6. Copy the **Connection URI** from the instance card or instance details page.

### Add Neo4j credentials to Colab

In Colab **Secrets**, add:

- `NEO4J_URI`
- `NEO4J_USERNAME`
- `NEO4J_PASSWORD`

Then set:

```python
USE_REAL_NEO4J = True
```

If you are not using Colab Secrets, set `PROMPT_FOR_NEO4J_CREDENTIALS = True` and paste the values into the hidden prompts.

Recommended classroom setup:

- Easiest: each student creates their own AuraDB Free instance.
- Instructor-controlled: the instructor seeds one database and gives students limited query credentials.
- A small database, not a production database.

In [ ]:
USE_REAL_NEO4J = False  # Change to True only when you have a Neo4j Aura database.
PROMPT_FOR_NEO4J_CREDENTIALS = False  # Keep False for Run all; set True to ask for hidden prompts.


def try_load_neo4j_credentials(prompt_if_missing=False):
    global NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD

    def read_secret(name):
        value = os.environ.get(name)
        if value:
            return value
        value = _load_colab_secret(name)
        if value:
            os.environ[name] = value
            return value
        return None

    uri = read_secret("NEO4J_URI")
    username = read_secret("NEO4J_USERNAME")
    password = read_secret("NEO4J_PASSWORD")

    if uri and username and password:
        NEO4J_URI = uri
        NEO4J_USERNAME = username
        NEO4J_PASSWORD = password
        return True

    if not prompt_if_missing:
        missing = [
            name
            for name, value in {
                "NEO4J_URI": uri,
                "NEO4J_USERNAME": username,
                "NEO4J_PASSWORD": password,
            }.items()
            if not value
        ]
        print("Neo4j credentials are not complete. Missing:", ", ".join(missing))
        return False

    uri = uri or input("Neo4j URI, for example neo4j+s://xxxx.databases.neo4j.io: ").strip()
    username = username or input("Neo4j username, often neo4j: ").strip()
    password = password or getpass.getpass("Neo4j password: ")

    if not (uri and username and password):
        print("Neo4j credentials are incomplete.")
        return False

    os.environ["NEO4J_URI"] = uri
    os.environ["NEO4J_USERNAME"] = username
    os.environ["NEO4J_PASSWORD"] = password
    NEO4J_URI = uri
    NEO4J_USERNAME = username
    NEO4J_PASSWORD = password
    return True


def node_statement(label, rows):
    return (
        f"UNWIND $rows AS row MERGE (n:{label} {{name: row.name}}) SET n += row",
        {"rows": rows},
    )


def relationship_statement(rel_type, rows):
    payload = [
        {"source": source, "target": target, "props": {**props, "course": COURSE_TAG}}
        for source, target, props in rows
    ]
    return (
        f'''
UNWIND $rows AS row
MATCH (a {{name: row.source, course: $course}})
MATCH (b {{name: row.target, course: $course}})
MERGE (a)-[r:{rel_type}]->(b)
SET r += row.props
''',
        {"rows": payload, "course": COURSE_TAG},
    )


SEED_STATEMENTS = [
    ("MATCH (n {course: $course}) DETACH DELETE n", {"course": COURSE_TAG}),
    node_statement("Region", REGIONS),
    node_statement("Policy", POLICIES),
    node_statement("Organization", ORGANIZATIONS),
    node_statement("Person", PEOPLE),
    node_statement("Event", EVENTS),
    node_statement("Tool", TOOLS),
]

for rel_type, rows in RELATIONSHIPS.items():
    SEED_STATEMENTS.append(relationship_statement(rel_type, rows))


def seed_neo4j(uri, username, password):
    from neo4j import GraphDatabase

    driver = GraphDatabase.driver(uri, auth=(username, password))
    try:
        with driver.session() as session:
            for statement, params in SEED_STATEMENTS:
                session.run(statement, **params)
    finally:
        driver.close()


if USE_REAL_NEO4J and try_load_neo4j_credentials(prompt_if_missing=PROMPT_FOR_NEO4J_CREDENTIALS):
    seed_neo4j(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD)
    print("Neo4j graph seeded.")
else:
    print("Skipping Neo4j seeding. Use Colab Secrets or set PROMPT_FOR_NEO4J_CREDENTIALS = True to run the real Neo4j path.")

## 7. LLM-first QA against the real Neo4j database

This cell uses the same LLM-first logic as the mock demo, but executes the generated Cypher against Neo4j. It works with either Gemini or OpenAI because it uses the provider wrapper defined earlier.

Keep `USE_REAL_NEO4J = False` if you only want the classroom mock path.

In [ ]:
def execute_neo4j_cypher(cypher, limit=20):
    from neo4j import GraphDatabase

    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    try:
        with driver.session() as session:
            result = session.run(cypher)
            rows = [dict(record) for record in result]
    finally:
        driver.close()
    return rows[:limit]


def graph_qa_real_neo4j(question, prompt_for_key=PROMPT_FOR_API_KEY):
    if not try_load_neo4j_credentials(prompt_if_missing=PROMPT_FOR_NEO4J_CREDENTIALS):
        raise ValueError("Neo4j credentials are not available.")
    if not resolve_llm_provider(prompt_if_missing=prompt_for_key):
        raise ValueError("No Gemini or OpenAI API key is available.")

    cypher = llm_generate_cypher(question)
    if not looks_read_only(cypher):
        raise ValueError(f"Generated Cypher did not pass the read-only guardrail: {cypher}")
    rows = execute_neo4j_cypher(cypher)
    answer = llm_summarize_graph_result(question, cypher, rows)
    return {"mode": "llm_cypher_with_real_neo4j", "question": question, "cypher": cypher, "rows": rows, "answer": answer}


if USE_REAL_NEO4J:
    try:
        real_result = graph_qa_real_neo4j("Which policies affect Harbor District?")
        print("MODE:", real_result["mode"])
        print("\nGENERATED CYPHER:\n" + real_result["cypher"])
        print("\nNEO4J RESULT:")
        display(pd.DataFrame(real_result["rows"]))
        print("ANSWER:", real_result["answer"])
    except Exception as exc:
        print(f"Real Neo4j QA skipped: {exc}")
else:
    print("Skipping real Neo4j QA because USE_REAL_NEO4J = False.")

## 8. Optional LangChain path: GraphCypherQAChain

This is an additional implementation path for students who want to see LangChain's built-in chain. It currently uses OpenAI through `langchain_openai`. The previous cell is the simpler Gemini/OpenAI-compatible path.

Important safety note: an LLM-generated database query can be wrong or unsafe. For teaching, use a sandbox database. For production, use least-privilege credentials, read-only permissions, query review, and strict guardrails.

In [ ]:
USE_LANGCHAIN_CHAIN = USE_LLM_REASONING  # The chain runs only if Neo4j credentials and an OpenAI API key are available.

if USE_REAL_NEO4J and USE_LANGCHAIN_CHAIN and try_load_neo4j_credentials(prompt_if_missing=PROMPT_FOR_NEO4J_CREDENTIALS) and try_load_provider_key("openai", prompt_if_missing=PROMPT_FOR_API_KEY):
    from langchain_neo4j import GraphCypherQAChain, Neo4jGraph
    from langchain_openai import ChatOpenAI

    graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD)
    graph.refresh_schema()
    print(graph.schema)

    llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)
    chain = GraphCypherQAChain.from_llm(
        llm=llm,
        graph=graph,
        verbose=True,
        return_intermediate_steps=True,
        allow_dangerous_requests=True,
    )

    response = chain.invoke({"query": "Which policies affect Harbor District?"})
    response
else:
    print("Skipping real Neo4j + LangChain execution. This optional LangChain cell uses OpenAI; provide Neo4j credentials and OPENAI_API_KEY, or use the earlier Gemini/OpenAI LLM-first demo.")

## 9. A simple query guardrail

This does not replace database permissions. It gives students a concrete way to think about risk.

In [ ]:
DANGEROUS_CYPHER = re.compile(
    r"\b(CREATE|MERGE|DELETE|DETACH|SET|REMOVE|DROP|ALTER|LOAD\s+CSV|CALL\s+dbms)\b",
    re.IGNORECASE,
)


def looks_read_only(cypher):
    text = cypher.strip()
    if DANGEROUS_CYPHER.search(text):
        return False
    return text.upper().startswith(("MATCH", "WITH", "RETURN", "CALL DB."))


queries_to_check = [
    "MATCH (p:Policy)-[:AFFECTS]->(r:Region) RETURN p.name, r.name",
    "MATCH (n) DETACH DELETE n",
    "MERGE (p:Policy {name: 'New Policy'}) RETURN p",
]

for query in queries_to_check:
    print(f"{str(looks_read_only(query)):5} | {query}")

## 10. Student tasks

1. Ask: **Which organizations coordinate with the Emergency Agency?**  
   Write the Cypher manually first. Then compare with the LLM-generated Cypher if you have the real path enabled.

2. Ask: **Which event triggered the Road Closure Plan?**  
   Explain why this is a path query rather than a keyword search.

3. Ask: **Which AI tool is connected to misinformation response?**  
   Inspect whether the generated query uses the relationship `USES_AI_TOOL`.

4. Safety check: create one unsafe Cypher query and one safe Cypher query. Test both with `looks_read_only`.

5. Reflection: name one question that is easy for text-to-Cypher and one question that is hard. Explain why.

## References

- Google AI Studio Gemini API key guide: https://ai.google.dev/gemini-api/docs/api-key
- Neo4j AuraDB creating an instance: https://neo4j.com/docs/aura/auradb/getting-started/create-database/
- Neo4j AuraDB connecting to an instance: https://neo4j.com/docs/aura/auradb/getting-started/connect-database/
- LangChain Neo4j graph integration: https://docs.langchain.com/oss/python/integrations/graphs/neo4j_cypher
- LangChain Neo4j provider page: https://docs.langchain.com/oss/python/integrations/providers/neo4j
- Neo4j LangChain developer guide: https://neo4j.com/developer/genai-ecosystem/langchain/